# soinfer Colab bootstrap

Runs `sparse-offload-infer` on a real GPU. This dev machine has no NVIDIA hardware (see `docs/DESIGN.md`), so every CUDA build/test/bench step happens here instead.

Runtime -> Change runtime type -> **T4 GPU** before running any cell below.

Fill in `REPO_URL` once the repo has a GitHub remote, then run the cells top to bottom.

In [ ]:
REPO_URL = "https://github.com/Venkateswarrao9025/sparse-offload-infer.git"
REPO_DIR = "sparse-offload-infer"

**The repo is private.** Plain `git clone {REPO_URL}` will fail with no credential. Either:
- clone with a token inline: `https://<token>@github.com/Venkateswarrao9025/sparse-offload-infer.git` (paste a GitHub PAT with `repo` scope when you fill in `REPO_URL`, don't leave it in a saved/shared copy of this notebook), or
- run `from google.colab import userdata` and store a token as a Colab secret, then build the URL from it in code instead of pasting it in plaintext above.

In [ ]:
!nvidia-smi

In [ ]:
import os

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull

## Build

`--no-build-isolation` links against Colab's preinstalled torch+CUDA pairing instead of pip resolving its own in an isolated env (see `docs/DESIGN.md`).

In [ ]:
!python -c "import torch; print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))"

In [ ]:
!make build

## Test

In [ ]:
!make test

## Benchmark

Writes `reports/m0_pcie_bandwidth.csv` and `reports/m0_baseline.csv`. Lock GPU clocks first if you have the permissions to (`nvidia-smi -lgc`); Colab T4 instances usually don't allow this, note it if so instead of pretending you did.

In [ ]:
!nvidia-smi -lgc $(nvidia-smi --query-gpu=clocks.max.sm --format=csv,noheader,nounits) 2>&1 || echo 'clock lock not permitted on this instance -- noted, proceeding without it'

In [ ]:
!make bench

## Get results back to the local repo

Colab's git identity/credentials are not the ones set up on the local machine, so this pushes commits under whatever account you authenticate as here -- do that deliberately rather than on autopilot. Simplest safe option: download the CSVs and commit them locally instead.

In [ ]:
from google.colab import files

files.download("reports/m0_pcie_bandwidth.csv")
files.download("reports/m0_baseline.csv")